# NóiLái — E4 skeleton: tone/onset/rime probes and patching in Gemma 3 (Colab / Kaggle 2×T4)

Loads **Gemma 3 1B (IT)** in float32 on one T4 (fp32 ≈ 4 GB), or **Gemma 3 4B (IT)** as the
text-only `Gemma3ForCausalLM` (skips the SigLIP tower) **sharded over 2×T4** with
`device_map="auto"` (fp32 ≈ 15.5 GB does not fit one T4; DESIGN_DECISIONS 9.5). Gemma 3 must
never run in fp16; bf16 only on a bf16-capable GPU (L4/A100) and then with the 200-pair drift
check. The notebook extracts residual-stream states at each syllable's last sub-token under
NFC and NFD with `noilai.probe.extract`, fits layer-wise probes with control tasks
(`noilai.probe.probes`), and holds the patching skeleton (`noilai.probe.patching`) for the
DESIGN_DECISIONS 9.3 pair *bí mà* (clean) vs *bí mạ* (corrupt) under V3 with E1's frozen
prompt. Results go to Drive (Colab) or the working directory (Kaggle). Secrets: `HF_TOKEN`
(gated Gemma), `GITHUB_TOKEN` if no bundle.

The two cells tagged `e4` are the experiment and are meant to be edited as E4 takes shape
(plan §2.6, weeks of Nov 23 – Dec 6; Gate 4 keeps probing only if patching is clean).

In [ ]:
# ---- parameters: edit this cell only; nothing below asks for input ------------------------------
import time
SESSION_T0 = time.time()

REPO_URL = "https://github.com/OWNER/REPO.git"     # [UNCERTAIN: verify]
REPO_REF = "main"
REPO_SUBDIR = "noilai"
DRIVE_DIR = "/content/drive/MyDrive/noilai"         # bundle, release and outputs live here
BUNDLE_PATH = DRIVE_DIR + "/noilai-main.bundle"
CLONE_DIR = "/content/repo"
WORK_DIR = "/content"
HF_HOME = "/content/hf"
ITEMS_DATASET_DIR = None                            # probes use the syllable inventory, not the item files
VERIFY_ITEM_KEYS = []
OUT_DIR = DRIVE_DIR + "/probe"

VLLM_VERSION = None                                 # not used here: probes run through transformers
TRANSFORMERS_VERSION = None
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
HF_TOKEN_SECRET = "HF_TOKEN"

MODEL_CONFIG = "gemma-3-1b-it"    # entry of configs/models.yaml (hf_id read from it); "gemma-3-4b-it" needs N_GPUS = 2 (Kaggle 2xT4, fp32 sharded)
DTYPE = "float32"                 # "bfloat16" only on a bf16-capable GPU; never "float16" for Gemma 3
TEXT_ONLY = True                  # multimodal checkpoints (4B/12B): load Gemma3ForCausalLM, the text tower only (DD 9.5)
N_SYLLABLES = 600                 # tone-bearing syllables sampled from the inventory (train/test split by syllable)
ENCODINGS = ["nfc", "nfd"]
FEATURES = ["tone", "onset", "rime"]
POSITION = "last"                 # "last" sub-token of the syllable or "after"
SEED = 20261004
BATCH_SIZE = 8

PLATFORM, GPU_TYPE, N_GPUS = "colab", "t4", 1    # Kaggle 2xT4 for the 4B: PLATFORM "kaggle", N_GPUS 2, DRIVE_DIR -> /kaggle/working/noilai
REQUIRE_GPU = True
RUN_LABEL = "E4_probe"

In [ ]:
# secrets: environment -> Colab userdata -> Kaggle Secrets. Values go into os.environ for the
# library that needs them and are reported as found / not found. Nothing here prints a value.
import os

def _secret(name):
    val = os.environ.get(name)
    if val:
        return val
    try:
        from google.colab import userdata          # Colab
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle
        try:
            val = UserSecretsClient().get_secret(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    return None

def _export(name, required=False):
    val = _secret(name)
    if val:
        os.environ[name] = val
        print(f"secret {name}: found ({len(val)} chars, not shown)")
        return True
    print(f"secret {name}: not found")
    if required:
        raise SystemExit(f"{name} is required for this notebook")
    return False

In [ ]:
# Drive: results, the bundle and (optionally) the release live under DRIVE_DIR
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
    print("Drive mounted; outputs ->", OUT_DIR)
except ImportError:
    print("not on Colab: outputs stay local under", WORK_DIR)
    OUT_DIR = str(Path(WORK_DIR) / "probe")
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# (a) environment check: interpreter, GPU(s), CUDA, vLLM's compute-capability floor (7.5), disk
import json, shutil, subprocess, sys
print(sys.version.split()[0], "on", sys.platform)
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi not found: is a GPU accelerator attached?")
ENV_CHECK = {"n_gpus": 0, "devices": [], "vllm_ok": False}
try:
    import torch
    print("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())
    for i in range(torch.cuda.device_count()):
        cap = torch.cuda.get_device_capability(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 2**30
        ENV_CHECK["devices"].append({"name": torch.cuda.get_device_name(i), "capability": f"{cap[0]}.{cap[1]}", "memory_gb": round(mem, 1)})
        print(f"  gpu{i}: {torch.cuda.get_device_name(i)}  capability {cap[0]}.{cap[1]}  {mem:.1f} GB")
    ENV_CHECK["n_gpus"] = torch.cuda.device_count()
    ENV_CHECK["vllm_ok"] = ENV_CHECK["n_gpus"] > 0 and all(
        tuple(int(x) for x in d["capability"].split(".")) >= (7, 5) for d in ENV_CHECK["devices"])
except ImportError:
    print("torch is not importable yet (installed in the install cell)")
if REQUIRE_GPU and ENV_CHECK["n_gpus"] == 0:
    raise SystemExit("no CUDA device: choose the GPU accelerator (Kaggle: 'GPU T4 x2') and restart")
if ENV_CHECK["n_gpus"] and not ENV_CHECK["vllm_ok"]:
    print("WARNING: a device is below compute capability 7.5 (P100?) -> vLLM cannot run; use backend hf / llama.cpp")
du = shutil.disk_usage(WORK_DIR)
print(f"disk at {WORK_DIR}: {du.free / 2**30:.1f} GB free of {du.total / 2**30:.1f} GB")
print(json.dumps(ENV_CHECK))

In [ ]:
# (b) clone: from a git bundle in a private dataset / Drive when present (no token needed), else from
# GitHub with a token from the platform's secret store. The token reaches git only through GIT_ASKPASS,
# a two-line helper that echoes an environment variable: it is never in a URL, an argv or this output.
import os, stat, subprocess, sys
from pathlib import Path

clone_dir = Path(CLONE_DIR)
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
bundle = Path(BUNDLE_PATH) if BUNDLE_PATH else None
if bundle is not None and not bundle.exists():
    print(f"bundle {bundle} not found; falling back to {REPO_URL}")
    bundle = None
if bundle is None:
    if "OWNER/REPO" in REPO_URL:
        raise SystemExit("set REPO_URL (or provide BUNDLE_PATH) in the parameters cell")
    if _export(GITHUB_TOKEN_SECRET):
        askpass = Path.home() / ".noilai_askpass.sh"
        askpass.write_text("#!/bin/sh\necho \"$" + GITHUB_TOKEN_SECRET + "\"\n")
        askpass.chmod(stat.S_IRWXU)
        env["GIT_ASKPASS"] = str(askpass)
source = str(bundle) if bundle else REPO_URL
if (clone_dir / ".git").exists():
    subprocess.run(["git", "-C", str(clone_dir), "fetch", "--tags", source, "+refs/heads/*:refs/remotes/origin/*"], check=True, env=env)
else:
    subprocess.run(["git", "clone", source, str(clone_dir)], check=True, env=env)
# a persisted clone may hold a STALE local branch named like REPO_REF: try the freshly fetched
# remote-tracking ref first and the literal ref last (commit hashes and tags have no remote form)
import re
cands = [REPO_REF] if re.fullmatch(r"[0-9a-f]{7,40}", REPO_REF) else [f"origin/{REPO_REF}", REPO_REF]
for cand in cands:
    if subprocess.run(["git", "-C", str(clone_dir), "checkout", "--detach", cand], env=env).returncode == 0:
        break
else:
    raise SystemExit(f"cannot check out {REPO_REF} (tried {cands})")
HEAD = subprocess.run(["git", "-C", str(clone_dir), "rev-parse", "HEAD"], check=True, env=env,
                      capture_output=True, text=True).stdout.strip()
PROJECT = clone_dir / REPO_SUBDIR if (clone_dir / REPO_SUBDIR / "pyproject.toml").exists() else clone_dir
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
print("HEAD", HEAD)
print("project", PROJECT)

In [ ]:
# (c) install: transformers path only (eval extra for torch/transformers/accelerate, dev extra for scikit-learn probes)
import subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[eval,dev]"], check=True)
os.environ["HF_HOME"] = HF_HOME
Path(HF_HOME).mkdir(parents=True, exist_ok=True)
_export(HF_TOKEN_SECRET)
import colab_setup as CS
ENV_RECORD = CS.record_environment(Path(OUT_DIR) / "env")
print("environment recorded at", ENV_RECORD)

In [ ]:
# (d) third-party resources (SHA-256 against data/HASHES.json) and the item file(s) of the runs this
# notebook will execute: the keys are DERIVED from the run ids (plus the smoke run) unless
# VERIFY_ITEM_KEYS lists them; a seeded sub-sample the plan derives from another file is written first
# (--materialize). Each file is checked for its hash against configs/run_plan.yaml, the header record,
# the canary on every row and its row/cell counts. A failure stops the notebook here.
import shutil, subprocess, sys
from pathlib import Path
import kaggle_run_plan as KRP
r = subprocess.run([sys.executable, "scripts/fetch_resources.py"], check=False)
if r.returncode:
    raise SystemExit("resource fetch or hash check failed (see above)")
if ITEMS_DATASET_DIR:
    src = Path(ITEMS_DATASET_DIR)
    if not src.exists():
        raise SystemExit(f"ITEMS_DATASET_DIR {src} not found: attach the private dataset that holds data/release")
    rel = Path(KRP.load_plan()["release"])            # e.g. data/release/v0.2: the plan's `release` key decides where the files go
    dest = PROJECT / rel
    if (src / rel.name).exists():                      # the dataset holds the data/release tree (one directory per version)
        shutil.copytree(src / rel.name, dest, dirs_exist_ok=True)
    elif (src / "manifest.json").exists():             # the dataset holds ONE release directory
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        raise SystemExit(f"{src} holds neither {rel.name}/ nor a release manifest.json: attach the frozen release")
    print("release copied from", src, "to", dest)
_run_ids = list(globals().get("RUN_IDS") or [globals().get("RUN_ID")]) + list(globals().get("SMOKE_RUN_IDS") or [])
KEYS = VERIFY_ITEM_KEYS or KRP.item_keys_for_runs(KRP.load_plan(), [r for r in _run_ids if r])
print("item files to verify:", KEYS)
for key in KEYS:
    r = subprocess.run([sys.executable, "scripts/kaggle_verify_items.py", "--key", key, "--materialize"], check=False)
    if r.returncode:
        raise SystemExit(f"item file {key} failed verification; nothing runs on it")

In [ ]:
# load the model named by MODEL_CONFIG in DTYPE (fp32 by default: Gemma 3 overflows in fp16). The 4B/12B
# checkpoints are multimodal: TEXT_ONLY loads Gemma3ForCausalLM (the text tower, no SigLIP), and N_GPUS > 1
# shards it over the session's GPUs with device_map="auto" (accelerate); DESIGN_DECISIONS 9.5.
import re, torch, yaml
from transformers import AutoModelForCausalLM, AutoTokenizer
entry = next(m for m in yaml.safe_load(open("configs/models.yaml", encoding="utf-8"))["models"] if m["name"] == MODEL_CONFIG)
is_gemma3 = entry.get("family") == "gemma3" or bool(re.match(r"google/gemma-3-", entry.get("hf_id") or ""))
if is_gemma3 and DTYPE == "float16":
    raise SystemExit("Gemma 3 must not run in float16 (transformers PR #36832)")
if DTYPE == "bfloat16" and torch.cuda.is_available() and torch.cuda.get_device_capability(0) < (8, 0):
    raise SystemExit("this GPU has no bfloat16 units; use float32 (or an L4/A100 runtime)")
n_visible = torch.cuda.device_count() if torch.cuda.is_available() else 0
if n_visible < N_GPUS:
    raise SystemExit(f"N_GPUS = {N_GPUS} but {n_visible} CUDA device(s) are visible: the 4B in fp32 needs Kaggle's 2xT4")
HF_ID = entry["hf_id"]
REVISION = entry.get("revision")        # a full commit hash before a paper run (DD 7.1); None = the hub's default branch
print("loading", HF_ID, "in", DTYPE, "| hf_id_status:", entry.get("hf_id_status"), "| revision:", REVISION, "| gpus:", N_GPUS)
tok = AutoTokenizer.from_pretrained(HF_ID, revision=REVISION)
device_map = "auto" if N_GPUS > 1 else ("cuda" if torch.cuda.is_available() else "cpu")
load_kw = {"dtype": getattr(torch, DTYPE), "device_map": device_map, "revision": REVISION}   # transformers 5.x: `dtype`, not torch_dtype
multimodal = is_gemma3 and float(entry.get("params_b") or 0) > 1.5
if multimodal and TEXT_ONLY:
    from transformers import Gemma3ForCausalLM
    model = Gemma3ForCausalLM.from_pretrained(HF_ID, **load_kw).eval()
else:
    model = AutoModelForCausalLM.from_pretrained(HF_ID, **load_kw).eval()
cfg = getattr(model.config, "text_config", model.config)
print("class:", type(model).__name__, "| layers:", cfg.num_hidden_layers, "| hidden:", cfg.hidden_size,
      "| devices:", sorted({str(p.device) for p in model.parameters()}))

### E4 CELL — layer-wise probes with control tasks
Tone-bearing syllables from the inventory are placed in carrier sentences (NFC and NFD), the
residual stream at the syllable's last sub-token is extracted at every layer, and a probe per
layer predicts tone / onset / rime with a syllable-disjoint split; selectivity = real − control.
Edit freely as E4 develops; this is the skeleton called for by the plan.

In [ ]:
# E4 CELL: probes. Uses noilai.probe as it exists in the clone (extract.make_examples,
# extract.extract_hidden_states, probes.run_layer_probes, probes.results_table).
import json, random
import numpy as np
import pandas as pd
from pathlib import Path
from noilai.probe import extract, probes
from noilai.vi import lexicon as L
from noilai.vi.syllable import try_parse

rng = random.Random(SEED)
pool = [s for s in L.load_hunspell_syllables("new") if (p := try_parse(s)) is not None and p.syllable.tone != 0]
rng.shuffle(pool)
syllables = pool[:N_SYLLABLES]
PROBE_TABLES = {}
for enc in ENCODINGS:
    exs = extract.make_examples(syllables, encoding=enc)
    H = extract.extract_hidden_states(model, tok, exs, batch_size=BATCH_SIZE, positions=(POSITION,))[POSITION]
    groups = [e.syllable for e in exs]
    for feature in [f for f in FEATURES if f in exs[0].labels]:
        labels = [e.labels[feature] for e in exs]
        res = probes.run_layer_probes(H, labels, groups, feature=feature, seed=SEED)
        table = probes.results_table(res)
        PROBE_TABLES[(enc, feature)] = table
        best = probes.best_layer(res)
        print(f"{enc:>3} {feature:<6} best layer {best.layer}: {table.loc[table['layer'] == best.layer].to_dict('records')[0]}")
        Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
        table.to_csv(Path(OUT_DIR) / f"probe_{MODEL_CONFIG}_{enc}_{feature}_{POSITION}.csv", index=False)
    np.save(Path(OUT_DIR) / f"hidden_{MODEL_CONFIG}_{enc}_{POSITION}.npy", H.astype(np.float16))
json.dump({"model": HF_ID, "dtype": DTYPE, "n_syllables": len(syllables), "encodings": ENCODINGS, "features": FEATURES,
           "position": POSITION, "seed": SEED, "head": HEAD}, open(Path(OUT_DIR) / f"probe_{MODEL_CONFIG}_manifest.json", "w"), indent=2)

### E4 CELL — activation patching (DESIGN_DECISIONS 9.3 pair *bí mà* / *bí mạ*, readout B)
The varying syllable is the SECOND input syllable, so under V3 (swap tones) the two gold
answers diverge at their FIRST syllable: *bí mà* → *bì má* (clean) and *bí mạ* → *bị má*
(corrupt), read at the pieces `▁bì` vs `▁bị`. Every output is recomputed by the rule engine
(`noilai.gen.variants`) and checked for legality; the prompt is E1's frozen V3 template with
its three rule-engine demonstrations and the IT chat template, the answer teacher-forced from
the `Đáp án:` line. (The plan's *bí mật* / *bí mất* is not a legal V3 pair — equal tones make V3
the identity — and belongs to V4 patching: *bí mật* → *bật mí*.) The cell caches the clean run,
patches the corrupt run layer by layer at the varying syllable's positions and reports recovery
of the logit difference; it refuses a pair whose token spans differ outside that syllable.

In [ ]:
# E4 CELL: activation patching on the DESIGN_DECISIONS 9.3 pair with E1's exact V3 prompt (frozen
# template p0, three rule-engine demonstrations, the IT chat template): readout B. The pair, its outputs
# and their legality come from the rule engine; nothing below is a typed-out example. [NATIVE-CHECK] the
# three syllables are ordinary (bí "gourd/secret", mà "but", mạ "rice seedling"); the outputs are pseudo-phrases.
import json, torch
from pathlib import Path
from noilai.eval import prompts as P
from noilai.gen import variants as V
from noilai.probe import patching
from noilai.vi import lexicon as L
from noilai.vi.syllable import parse, spell

VARIANT = "V3"
PARTNER, CLEAN_SYL, CORRUPT_SYL = "bí", "mà", "mạ"   # the varying syllable is the SECOND one
inv = L.load_inventory()

def _pair(second):
    a, b = parse(PARTNER).syllable, parse(second).syllable
    if V.is_identity(VARIANT, a, b):
        raise SystemExit(f"{PARTNER} {second} is an identity under {VARIANT}: not a legal patching pair (DD 3.4)")
    out = V.apply(VARIANT, a, b)
    if not all(inv.is_legal(x, "onset_rime") for x in out):
        raise SystemExit(f"{VARIANT} on {PARTNER} {second} gives an illegal syllable")
    return f"{PARTNER} {second}", f"{spell(out[0])} {spell(out[1])}"

CLEAN, ANSWER_CLEAN = _pair(CLEAN_SYL)
CORRUPT, ANSWER_CORRUPT = _pair(CORRUPT_SYL)
first_clean, first_corrupt = ANSWER_CLEAN.split()[0], ANSWER_CORRUPT.split()[0]
assert first_clean != first_corrupt, "the two gold answers must diverge at their first syllable"
print(f"clean {CLEAN} -> {ANSWER_CLEAN} | corrupt {CORRUPT} -> {ANSWER_CORRUPT} | readout {first_clean} vs {first_corrupt}")

def _prompt(phrase):
    item = {"task": "T1", "variant": VARIANT, "input": phrase, "item_id": "P-" + phrase.replace(" ", "_"), "gold": [""]}
    msgs = P.render(item, paraphrase="p0", shots=3, arm="nfc")          # E1's frozen template + rule-engine demonstrations
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + P.answer_marker()   # teacher-forced answer line
    enc = tok(text, return_offsets_mapping=True, return_tensors="pt", add_special_tokens=False)          # the template carries <bos>: no double BOS
    start = text.rindex(phrase) + len(PARTNER) + 1                        # the ITEM's varying syllable (the demonstrations come earlier)
    end = start + len(phrase.split()[1])
    positions = [i for i, (a, b) in enumerate(enc["offset_mapping"][0].tolist()) if b > start and a < end]
    return enc["input_ids"], positions, text

clean_ids, clean_pos, clean_text = _prompt(CLEAN)
corrupt_ids, corrupt_pos, _ = _prompt(CORRUPT)
tok_clean = tok(" " + first_clean, add_special_tokens=False)["input_ids"]       # one word-initial piece each (DD 9.3)
tok_corrupt = tok(" " + first_corrupt, add_special_tokens=False)["input_ids"]
if clean_ids.shape != corrupt_ids.shape or clean_pos != corrupt_pos or len(tok_clean) != 1 or len(tok_corrupt) != 1:
    print("pair not aligned for this tokenizer: token spans or answer pieces differ; pick another DD 9.3 pair",
          clean_ids.shape, corrupt_ids.shape, clean_pos, corrupt_pos, tok_clean, tok_corrupt)
else:
    diff = (clean_ids[0] != corrupt_ids[0]).nonzero().flatten().tolist()
    assert set(diff) <= set(clean_pos), f"clean/corrupt prompts differ outside the varying syllable: {diff} vs {clean_pos}"
    device = next(model.parameters()).device
    result = patching.run_patching(model, clean_ids.to(device), corrupt_ids.to(device), answer_pos=clean_ids.shape[1] - 1,
                                   tok_clean=tok_clean[0], tok_corrupt=tok_corrupt[0], position_groups=[clean_pos])
    print(result)
    json.dump({"variant": VARIANT, "clean": CLEAN, "corrupt": CORRUPT, "answer_clean": ANSWER_CLEAN, "answer_corrupt": ANSWER_CORRUPT,
               "positions": clean_pos, "prompt_hash": P.prompt_hash([{"role": "user", "content": clean_text}]),
               "result": str(result), "head": HEAD, "model": HF_ID, "dtype": DTYPE},
              open(Path(OUT_DIR) / f"patching_{MODEL_CONFIG}_{VARIANT}_{PARTNER}_{CLEAN_SYL}_{CORRUPT_SYL}.json", "w"), ensure_ascii=False, indent=2)

In [ ]:
# GPU-hours log (checklist C1): this session as one E4 row; totals printed and copied to Drive
import shutil
import compute_log as CL
LOG = PROJECT / "data" / "compute_log.csv"
CL.append_entry(LOG, CL.Entry(date=CL.today(), platform=PLATFORM, gpu_type=GPU_TYPE, n_gpus=N_GPUS,
                              hours=CL.hours_since(SESSION_T0), run_id=f"{RUN_LABEL}__{MODEL_CONFIG}", purpose="E4_probe"))
print(CL.format_totals(CL.totals(CL.read_entries(LOG))))
shutil.copy2(LOG, Path(OUT_DIR) / "compute_log.csv")